# Week 2 Day 2: Guided Practice SOLUTIONS - Heart Disease Classification

**Dataset:** Heart Disease UCI  
**Goal:** Practice evaluating classification models with instructor support  
**Scaffolding Level:** 70% provided (guided practice)  
**Time Estimate:** 20 minutes (in-class)

---

## How to Use This Notebook

This is the **SOLUTIONS** version with all answers filled in.

✅ Use this to:
- Check your work after completing the guided practice
- Review correct interpretations of confusion matrix
- See example threshold analysis

---

## Learning Objectives

By completing this exercise, you will:
1. Interpret confusion matrix values (TP, TN, FP, FN)
2. Calculate metrics manually from confusion matrix
3. Compare different decision thresholds
4. Choose appropriate threshold based on business costs

---

## The Challenge

You are a data scientist at a hospital. Your task is to evaluate a classifier that predicts whether a patient has heart disease.

**Business context:**
- **False Negative (FN):** Patient has heart disease but model says they don't → **DANGEROUS!** (no treatment)
- **False Positive (FP):** Patient doesn't have heart disease but model says they do → Costly but safer (extra tests)

**Key Question:** Which error is worse? **FN is worse!**

---

## Part 1: Setup and Data Loading

In [ ]:
# Import all necessary libraries

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

print("✅ All imports successful!")

In [ ]:
# Load Heart Disease dataset

column_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 
                'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

df = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data',
    names=column_names,
    na_values='?'
)

print("✅ Data loaded!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Prepare data

df['target'] = (df['target'] > 0).astype(int)
df = df.dropna()

print("Class distribution:")
print(df['target'].value_counts().sort_index())
print(f"\nClass balance:")
print(f"  No disease (0): {(df['target']==0).sum()} ({(df['target']==0).sum()/len(df)*100:.1f}%)")
print(f"  Disease (1):    {(df['target']==1).sum()} ({(df['target']==1).sum()/len(df)*100:.1f}%)")

In [ ]:
# Separate features and target

X = df.drop('target', axis=1).values
y = df['target'].values

print(f"✅ Features and target separated!")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

---

## Part 2: Data Preparation

In [ ]:
# Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("✅ Data split complete!")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nClass distribution in training set:")
print(f"  No disease: {(y_train==0).sum()}")
print(f"  Disease:    {(y_train==1).sum()}")

In [ ]:
# Feature scaling

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled!")
print(f"\nTraining data (first feature):")
print(f"  Mean: {X_train_scaled[:, 0].mean():.4f}")
print(f"  Std:  {X_train_scaled[:, 0].std():.4f}")

---

## Part 3: Model Training

In [ ]:
# Create and train model

model = LogisticRegression(max_iter=10000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)

print("✅ Model trained and predictions made!")
print(f"\nFirst 5 predictions:")
print(f"  Classes: {y_pred[:5]}")
print(f"  Probabilities (disease): {y_pred_proba[:5, 1]}")

---

## Part 4: Model Evaluation - SOLUTIONS

In [ ]:
# Generate confusion matrix

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print("\n" + "="*50)
print("CONFUSION MATRIX BREAKDOWN")
print("="*50)
print(f"  True Negatives (TN):  {tn}")
print(f"  False Positives (FP): {fp}")
print(f"  False Negatives (FN): {fn}")
print(f"  True Positives (TP):  {tp}")
print("="*50)

### SOLUTION: Interpret the Confusion Matrix

**Note:** Actual values will vary based on random split, but interpretations are the same.

**EXAMPLE ANSWERS (with typical values):**

1. **True Negatives (TN = ~25):** Number of patients who **did NOT have heart disease** and were correctly predicted as **not having disease**.

2. **False Positives (FP = ~5):** Number of patients who **did NOT have heart disease** but were incorrectly predicted to have **heart disease**.

3. **False Negatives (FN = ~3):** Number of patients who **had heart disease** but were incorrectly predicted as **not having disease**.
   - **Why this is dangerous:** These patients won't receive treatment for their heart disease, which could lead to serious health complications or death.

4. **True Positives (TP = ~28):** Number of patients who **had heart disease** and were correctly predicted to have **heart disease**.

---

**DISCUSSION ANSWER:** FN is worse than FP because:
- FN means missing a disease → patient doesn't get life-saving treatment
- FP means false alarm → patient gets extra tests (costly but safe)
- In medical contexts, we prefer to be cautious (accept more FP to minimize FN)

### SOLUTION: Calculate Metrics Manually

**Example with typical values (TN=25, FP=5, FN=3, TP=28):**

```
Accuracy = (28 + 25) / (28 + 25 + 5 + 3) = 53 / 61 = 0.869

Precision = 28 / (28 + 5) = 28 / 33 = 0.848

Recall = 28 / (28 + 3) = 28 / 31 = 0.903

F1 = 2 × (0.848 × 0.903) / (0.848 + 0.903) = 2 × 0.766 / 1.751 = 0.875
```

**Key Insights:**
- Recall (90.3%) is higher than precision (84.8%) → model catches most disease cases
- This is good for medical screening where we want to minimize FN
- Accuracy is high (86.9%), but we care more about recall in this context

In [ ]:
# Check calculations against sklearn

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("="*60)
print("MODEL PERFORMANCE METRICS")
print("="*60)
print(f"Accuracy:  {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f"Precision: {precision:.3f} ({precision*100:.1f}%)")
print(f"Recall:    {recall:.3f} ({recall*100:.1f}%)")
print(f"F1-Score:  {f1:.3f} ({f1*100:.1f}%)")
print("="*60)
print("\n✅ These should match your manual calculations!")

In [ ]:
# Visualize confusion matrix

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Disease', 'Disease'],
    yticklabels=['No Disease', 'Disease'],
    cbar_kws={'label': 'Count'}
)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix: Heart Disease Classification')
plt.tight_layout()
plt.show()

---

## Part 5: Threshold Tuning - SOLUTIONS

In [ ]:
# Compare three thresholds

thresholds = [0.3, 0.5, 0.7]

print("="*80)
print("THRESHOLD COMPARISON")
print("="*80)
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'FN Count':<12}")
print("-"*80)

for thresh in thresholds:
    y_pred_thresh = (y_pred_proba[:, 1] >= thresh).astype(int)
    
    prec = precision_score(y_test, y_pred_thresh)
    rec = recall_score(y_test, y_pred_thresh)
    f1_thresh = f1_score(y_test, y_pred_thresh)
    
    cm_thresh = confusion_matrix(y_test, y_pred_thresh)
    fn_count = cm_thresh[1, 0]
    
    print(f"{thresh:<12.1f} {prec:<12.3f} {rec:<12.3f} {f1_thresh:<12.3f} {fn_count:<12}")

print("="*80)

### SOLUTION: Analyze Threshold Results

**Typical pattern observed:**

1. **What happens to recall as threshold decreases from 0.7 → 0.5 → 0.3?**

   **Answer:** Recall **increases** as threshold decreases. Lower threshold = more predictions of "disease" = catch more actual disease cases = higher recall.

2. **What happens to precision as threshold decreases?**

   **Answer:** Precision **decreases** as threshold decreases. Lower threshold = more predictions of "disease" = more false alarms = lower precision.

3. **Which threshold has the FEWEST false negatives (FN)?**

   **Answer:** Threshold **0.3** has the fewest FN (typically 1-2, vs 3-4 for 0.5, vs 5-7 for 0.7).

4. **For heart disease screening, which threshold would you recommend? Why?**

   **Answer:** I would recommend **threshold 0.3** because it minimizes false negatives, which is critical in medical screening. Missing a heart disease case (FN) could be fatal, while a false alarm (FP) just means extra tests. The slightly lower precision is an acceptable tradeoff for catching more disease cases.

---

**DISCUSSION ANSWER - Cost Analysis:**

If FN costs $100K and FP costs $5K:

**Threshold 0.3:** ~2 FN × $100K + ~8 FP × $5K = $200K + $40K = **$240K total cost**

**Threshold 0.5:** ~3 FN × $100K + ~5 FP × $5K = $300K + $25K = **$325K total cost**

**Threshold 0.7:** ~6 FN × $100K + ~2 FP × $5K = $600K + $10K = **$610K total cost**

**Recommendation:** Threshold 0.3 minimizes total cost and saves lives!

---

## Part 6: ROC Curve

In [ ]:
# Plot ROC curve

fpr, tpr, _ = roc_curve(y_test, y_pred_proba[:, 1])
auc = roc_auc_score(y_test, y_pred_proba[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guessing')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve: Heart Disease Classification')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAUC Score: {auc:.3f}")
print("Interpretation: AUC > 0.8 is considered good!")

---

## 🎉 Congratulations!

You've reviewed the complete solutions for Week 2 Day 2 guided practice!

**Key Learning Points:**

1. **Confusion Matrix is the Foundation**
   - All metrics (accuracy, precision, recall, F1) come from TP, TN, FP, FN
   - Understanding the matrix is essential for choosing the right metric

2. **Context Determines Metric Choice**
   - Medical screening → Optimize recall (catch all disease cases)
   - Spam filtering → Optimize precision (don't lose important emails)
   - Balanced costs → F1 or accuracy may be appropriate

3. **Threshold Tuning is Powerful**
   - Default 0.5 is arbitrary
   - Lower threshold → Higher recall, lower precision
   - Higher threshold → Lower recall, higher precision
   - Choose based on business costs of FN vs FP

4. **Manual Calculations Build Intuition**
   - Calculating metrics by hand helps you understand what they mean
   - You can verify sklearn results and catch errors
   - Makes you a better data scientist!

---

## Next Steps

1. **Take the quiz** - Test your understanding (8 questions, 60% to pass)
2. **Complete post-class exercise** - Apply to Titanic dataset independently
3. **Review self-assessment** - Check your Week 2 mastery

---

**Excellent work! You're ready to apply these skills independently!** 🎓

---

*Week 2 Day 2 Guided Practice SOLUTIONS v1.0 | December 2025*